# 03 — Évaluation d'un run

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lrivals/SerreIA/blob/main/notebooks/03_evaluation.ipynb)

Évaluation d'un checkpoint existant (docs/03-classifieur.md §4–§6) : métriques par
source, matrices de confusion, Grad-CAM, export ONNX. `RUN_ID = None` prend le dernier
run de `runs/` (sur Colab, restauré depuis le Drive).

Sur Colab : *Exécution → Modifier le type d'exécution → GPU (T4)*.

## Paramètres

In [ ]:
RUN_ID = None  # identifiant MLflow (dossier runs/<RUN_ID>/) ; None : le plus récent
SUBSET = 0  # nb d'images par jeu (0 : tout)
DEVICE = None
GRADCAM_N = 32
ONNX_N = 256
DATASETS = ["index", "plantvillage", "ccmt", "plantdoc"]

## Environnement

In [ ]:
# Environnement (PC et Colab) : dépôt, données, sorties (docs/data-drive.md).
# PC : rien n'est installé ; un jeu absent est récupéré du Drive par rclone, sinon la cellule
# s'arrête avec la commande à lancer. Colab : clone du dépôt à REV, installation, données via
# le Drive monté (archives MyDrive/SerreIA/data/*.tar), sorties précédentes restaurées.
import logging
import os
import subprocess
import sys
from pathlib import Path

DRIVE_DIR = "/content/drive/MyDrive/SerreIA"  # Colab : data/*.tar et runs/ ; None : pas de Drive
REPO_URL = "https://github.com/lrivals/SerreIA.git"
REV = "main"
COLAB = "google.colab" in sys.modules

if COLAB:
    ROOT = Path("/content/SerreIA")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(ROOT)], check=True)
    subprocess.run(["git", "-C", str(ROOT), "fetch", "-q", "origin", REV], check=True)
    subprocess.run(["git", "-C", str(ROOT), "checkout", "-q", "--detach", "FETCH_HEAD"], check=True)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ROOT}[vision]"], check=True
    )
else:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from tools.notebooks import drive_data  # noqa: E402

if COLAB and DRIVE_DIR:
    drive_data.mount_drive()
    drive_data.restore_all(DRIVE_DIR)
drive_data.ensure(DATASETS, DRIVE_DIR if COLAB else None)

# Colab : tous les cœurs pour le chargement, base MLflow séparée de celle du PC.
ENV_SET = (
    [f"data.num_workers={os.cpu_count()}", f"mlflow.tracking_uri={drive_data.COLAB_MLFLOW}"]
    if COLAB
    else []
)

## Run

In [ ]:
from vision.train.train_classifier import pick_device

if RUN_ID is None:
    ckpts = sorted((ROOT / "runs").glob("*/best.pt"), key=lambda p: p.stat().st_mtime)
    if not ckpts:
        raise FileNotFoundError("aucun runs/*/best.pt : lancer 02_entrainement")
    RUN_ID = ckpts[-1].parent.name
print("RUN_ID =", RUN_ID)

## Évaluation in-domain et cross-domain (§4)

In [ ]:
import torch
from IPython.display import Image, Markdown, display

from vision.eval.cross_domain_eval import REPORTS_DIR, log_to_mlflow, run_eval, summary_table

CKPT = ROOT / "runs" / RUN_ID / "best.pt"
device = pick_device(DEVICE)
with torch.inference_mode():
    payload = run_eval(CKPT, device, SUBSET)
OUT = REPORTS_DIR / RUN_ID
log_to_mlflow(RUN_ID, payload, OUT)
display(Markdown(summary_table(payload["results"])))

In [ ]:
for png in sorted(OUT.glob("cm_*_sante.png")):
    print(png.name)
    display(Image(filename=png, width=360))

## Grad-CAM (§5)

In [ ]:
from vision.eval.gradcam import make_sheets

sheets = make_sheets(CKPT, n=GRADCAM_N, save_cams=True, device=device)
for png in sheets:
    if png.name.endswith("_sante.png"):
        print(png.name)
        display(Image(filename=png))

## Export ONNX (§6 : écart < 0,5 point)

In [ ]:
from vision.export.to_onnx import export_and_check

onnx_res = export_and_check(CKPT, ONNX_N)
worst = onnx_res["worst_disagree_pts"]
print(f"écart max {worst:.2f} pt : {'OK' if worst < 0.5 else 'ÉCHEC'}")
drive_data.sync_all(DRIVE_DIR)

In [ ]:
for png in sorted(OUT.glob("cm_*_classe.png")):
    print(png.name)
    display(Image(filename=png))